In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "catcher-rag-eval"
os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [ ]:
from pathlib import Path
import sys

def find_project_root() -> Path:
    curr = Path.cwd()
    for parent in [curr] + list(curr.parents):
        if (parent / "pyproject.toml").exists():
            return parent
    return curr

PROJECT_ROOT = find_project_root()
src_dir = str(PROJECT_ROOT / "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

print("PROJECT_ROOT:", PROJECT_ROOT)

# 1. 문서 로드 — self_report + saving_guides

In [ ]:
PDF_PATHS = [
    PROJECT_ROOT / "data/raw/pdf/self_report/consumer_spending_report_v2.pdf",
    PROJECT_ROOT / "data/raw/txt/saving_guides.txt",
]

In [ ]:
from langchain_community.document_loaders import PDFPlumberLoader, TextLoader

all_docs = []
for path in PDF_PATHS:
    if str(path).endswith(".pdf"):
        loader = PDFPlumberLoader(str(path))
    elif str(path).endswith(".txt"):
        loader = TextLoader(str(path), encoding="utf-8")
    else:
        continue
    docs = loader.load()
    for d in docs:
        d.metadata["source"] = path.name
    all_docs.extend(docs)

print("총 문서 수:", len(all_docs))
print(all_docs[0].page_content[:300])

# 2. 문서 split

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=120)
split_docs = text_splitter.split_documents(all_docs)
print(f"청킹 후 문서 수: {len(split_docs)}")

# 3. 임베딩 + 벡터 DB

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(split_docs, embeddings)
print(f"벡터 수: {vectorstore.index.ntotal}")

# 4. Retriever 설정

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# 5. 문서 내용 확인

In [ ]:
# 소스별 내용 확인
for source in ["consumer_spending_report_v2.pdf", "saving_guides.txt"]:
    docs_by_source = [d for d in all_docs if d.metadata.get("source") == source]
    if docs_by_source:
        print(f"\n=== {source} ===")
        print(docs_by_source[0].page_content[:400])

# 6. 질문 → context → 답변 생성

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, max_tokens=300)

def run_rag(q):
    docs = retriever.invoke(q)
    context_texts = [doc.page_content[:400] for doc in docs]
    answer = llm.invoke(
        f"""질문: {q}

아래 문서에 있는 내용만 사용해서 핵심 답변을 1~2문장으로 작성하세요.
문서에 없는 내용은 절대 추가하지 마세요.

문서:
{context_texts}"""
    ).content
    sources = [doc.metadata.get("source", "출처 없음") for doc in docs]
    return answer, context_texts, sources

def target(inputs: dict):
    q = inputs["question"]
    answer, contexts, sources = run_rag(q)
    return {"answer": answer, "contexts": contexts}

In [ ]:
# 단건 테스트
q = "소비자 지출 보고서에서 가장 많은 지출 카테고리는?"
answer, contexts, sources = run_rag(q)
print("답변:", answer)
print("출처:", sources)

# 7. LangSmith Dataset 생성

> ⚠️ 위 내용 확인 후 questions/ground_truths를 실제 문서 내용에 맞게 수정하세요.

In [ ]:
from langsmith import Client

client = Client()
dataset_name = "catcher-rag-selfReport-eval"

# ⚠️ 문서 내용 확인 후 수정하세요
questions = [
    "소비자 지출 보고서에서 가장 많은 지출 카테고리는?",
    "절약 가이드에서 가장 먼저 권장하는 실천 방법은?",
    "보고서에서 제시하는 소비 패턴 개선 방법은?",
    "식비 절약을 위한 구체적인 방법은?",
    "저축을 늘리기 위한 첫 번째 단계는?",
]

# ⚠️ 실제 문서 내용으로 수정 필요
ground_truths = [
    "(문서 내용 확인 후 작성)",
    "(문서 내용 확인 후 작성)",
    "(문서 내용 확인 후 작성)",
    "(문서 내용 확인 후 작성)",
    "(문서 내용 확인 후 작성)",
]

existing = [d for d in client.list_datasets() if d.name == dataset_name]
if existing:
    dataset = existing[0]
    print(f"기존 dataset 사용: {dataset.name}")
else:
    dataset = client.create_dataset(dataset_name=dataset_name, description="self_report + saving_guides RAG 평가")
    for q, gt in zip(questions, ground_truths):
        client.create_example(
            inputs={"question": q},
            outputs={"ground_truth": gt},
            dataset_id=dataset.id
        )
    print(f"새 dataset 생성: {dataset.name} ({len(questions)}개)")

# 8. Evaluator 정의

In [ ]:
judge_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def correctness_evaluator(run, example):
    answer = run.outputs["answer"]
    ground_truth = example.outputs["ground_truth"]
    prompt = f"""다음 답변이 정답과 얼마나 일치하는지 0~1 점수로 평가해줘.
정답: {ground_truth}
답변: {answer}
숫자 하나만 출력해."""
    score = judge_llm.invoke(prompt).content.strip()
    return {"key": "correctness", "score": float(score)}

def faithfulness_evaluator(run, example):
    answer = run.outputs["answer"]
    contexts = run.outputs["contexts"]
    prompt = f"""아래 답변이 문서에 있는 내용만 사용했는지 0~1로 평가해줘.
숫자 하나만 출력해.
문서: {contexts}
답변: {answer}"""
    score = judge_llm.invoke(prompt).content.strip()
    return {"key": "faithfulness", "score": float(score)}

def source_coverage_evaluator(run, example):
    """두 소스(report + guide)가 골고루 활용됐는가"""
    contexts = run.outputs["contexts"]
    combined = " ".join(contexts)
    has_report = "spending" in combined.lower() or "report" in combined.lower() or "지출" in combined
    has_guide = "절약" in combined or "guide" in combined.lower() or "가이드" in combined
    score = 1.0 if (has_report and has_guide) else 0.5 if (has_report or has_guide) else 0.0
    return {"key": "source_coverage", "score": score}

print("evaluator 3개 정의 완료")

# 9. evaluate() 실행 → LangSmith 반영

In [ ]:
from langsmith.evaluation import evaluate

evaluate(
    target,
    data=dataset_name,
    evaluators=[correctness_evaluator, faithfulness_evaluator, source_coverage_evaluator],
    experiment_prefix="selfReport-rag-v1"
)